In [25]:
from rdkit import Chem
from rdkit.Chem import rdFMCS, Draw
import os
import subprocess
import networkx as nx

# Paths
mol2_dir = "/home/raheelx/cphmd_walkthrough/mol2_Epik"
sdf_dir = "/home/raheelx/cphmd_walkthrough/sdf_Epik"

# Create output directory if it doesn't exist
os.makedirs(sdf_dir, exist_ok=True)

# Convert mol2 files to sdf using Open Babel CLI
def convert_mol2_to_sdf(mol2_dir, sdf_dir):
    for filename in sorted(os.listdir(mol2_dir)):
        if filename.endswith(".mol2"):
            mol2_path = os.path.join(mol2_dir, filename)
            sdf_path = os.path.join(sdf_dir, filename.replace(".mol2", ".sdf"))
            # Run obabel conversion
            subprocess.run(["obabel", mol2_path, "-O", sdf_path], check=True)

# Load sdf files using RDKit
def load_sdfs(sdf_dir):
    mols = []
    for filename in sorted(os.listdir(sdf_dir)):
        if filename.endswith(".sdf"):
            path = os.path.join(sdf_dir, filename)
            mol = Chem.MolFromMolFile(path, sanitize=True)
            if mol:
                mols.append((filename.replace(".sdf", ""), mol))
            else:
                print(f"Failed to load {filename}")
    return mols

# Convert mol2 to sdf
convert_mol2_to_sdf(mol2_dir, sdf_dir)

# Load converted sdf files
mols = load_sdfs(sdf_dir)

print(f"Loaded {len(mols)} molecules")
for name, mol in mols:
    print(name, mol is not None)

# Extract RDKit Mol objects only for MCS
rdkit_mols = [mol for name, mol in mols]

# Run MCS
mcs_result = rdFMCS.FindMCS(rdkit_mols,
                            timeout=60,
                            completeRingsOnly=True,
                            ringMatchesRingOnly=True,
                            threshold=0.9,
                            matchValences=True)

print("\nMCS SMARTS:", mcs_result.smartsString)

# Create a Mol object for the MCS pattern
common_core = Chem.MolFromSmarts(mcs_result.smartsString)

# Identify variable atoms outside the MCS core and cluster into sites
for name, mol in mols:
    matches = mol.GetSubstructMatch(common_core)
    if not matches:
        print(f"No MCS match found for {name}")
        continue

    core_atoms = set(matches)
    variable_atoms = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetIdx() not in core_atoms]

    # Build graph of variable atoms connected by bonds
    G = nx.Graph()
    G.add_nodes_from(variable_atoms)

    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtomIdx()
        a2 = bond.GetEndAtomIdx()
        if a1 in variable_atoms and a2 in variable_atoms:
            G.add_edge(a1, a2)

    # Find connected components in the variable atom subgraph
    variable_sites = list(nx.connected_components(G))

    print(f"\nMolecule: {name}")
    print(f"Number of variable sites: {len(variable_sites)}")
    for i, site in enumerate(variable_sites):
        print(f" Site {i}: Atom indices {sorted(site)}")


1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted


Loaded 6 molecules
riboflavin_1 True
riboflavin_2 True
riboflavin_3 True
riboflavin_4 True
riboflavin_5 True
riboflavin_6 True

MCS SMARTS: [#8&!R]-&!@[#6&!R](-&!@[#6&!R]-&!@[#6&!R]-&!@[#7&R])-&!@[#6&!R](-&!@[#8&!R])-&!@[#6&!R]-&!@[#8&!R]

Molecule: riboflavin_1
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_2
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_3
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_4
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_5
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5

*** Open Babel Warning  in ReadMolecule
  Failed to kekulize aromatic bonds in MOL2 file (title is Riboflavin)

1 molecule converted
